Мьютекс — это механизм взаимного исключения: он позволяет только одному потоку или процессу одновременно войти в критическую секцию и работать с общим ресурсом, чтобы не возникали гонки данных и порча состояния. Если два потока одновременно обновляют один и тот же счетчик, файл, строку в БД или кеш, результат может стать неверным. Мьютекс решает это просто: один взял — остальные ждут, пока он его не освободит

Redis способен проводить атомарные операции
В нем операции можно проводить быстро, с низкой задержкой
Он предоставляет общий доступ для процессов, они могут договориться, кто владеет блокировкой

Атомарная операция - операция, которая выполняется целиком или не выполняется вовсе, ее нельзя увидеть в промежуточном состоянии.

Кто первый атомарно создал ключ блокировки, тот и вошел в критическую секцию. Критическая секция - блок кода, в котором может возникнуть гонка данных.

Гонка данных - это когда несколько потоков одновременно меняют одни и те же данные, и результат становится непредсказуемым. Она ломает корректность программы - появляются баги, потеря данных, падения.

**Объект блокировки** — это объект-синхронизатор, который умеет `acquire()` и `release()`: он «закрывает» и «открывает» доступ к критической секции

**Redis-клиент в Python** — это объект, через который твой код отправляет команды в Redis: читать, писать, ставить ключи, делать `SET NX EX` и т.д..

**Захват блокировки** — это попытка получить право первым войти в критическую секцию; пока блокировка занята, другие ждут

Ключ — это **замок** (блокировка, точнее - ее состояние), а токен — **метка владельца (кто этой блокировкой владеет)**. Ключ есть - блокировка занята, ключа нет - блокировка свободна. Ключ отсылает на какой-то определяемый общий ресурс, а токен показывает, кто сейчас имеет право совершать операции с этим ресурсом. Когда процесс захватывает блокировку, он записывает в ключ свой уникальный токен; потом при освобождении проверяет, что в ключе всё ещё его токен, и только тогда удаляет ключ. Почему говорят «ключ принадлежит токену»
Потому что удалять блокировку имеет право только тот, кто её создал.  
Если TTL истёк и другой процесс уже захватил тот же ключ, старый владелец не должен случайно удалить чужую блокировку.
### Анализ кода

\_\_init__ - создает объект блокировки, запоминает redis-клиент, имя замка, время жизни, паузу между попытками. Нужна, чтобы настроить мьютекс один раз и потом использовать его в with

\_\_enter__ - пытается захватить блокировку. Генерирует уникальный токен, в цикле делает атомарный SET. Нужна для того, чтобы код внутри with mu: выполнялся только после захвата лока

\_\_exit__ - Освобождает блокировку, но только если ключ в Redis всё ещё принадлежит этому токену. Нужна, чтобы случайно не удалить чужой лок, если TTL истёк и ключ уже перехватил другой процесс.

function - защищенная критическая секция. Без mutex тут была бы race condition.

main - создает 10 задач в 5 потоках и ждет их завершения, чтобы проверить, что mutex делает инкремент безопасным. Ответ должен быть 10.

In [ ]:
def __enter__(self):
        # Генерируем уникальный токен для текущего захвата
        self.token = str(uuid.uuid4())
        while True:
            # Атомарная операция: установить если нет (NX) с временем жизни (EX)
            if self.redis.set(self.lock_name, self.token, nx=True, ex=self.expire):
                return self
            # Если не удалось захватить — ждем и пробуем снова
            time.sleep(self.retry_delay)

uuid.uuid4() - создает случайный UUID версии 4, str - превращает его в строку. **Зачем это нужно?** Когда вы захватываете блокировку, нужно знать, кто именно её захватил. Потом в `__exit__` вы проверите, что освобождаете блокировку тем же самым «токеном», чтобы случайно не удалить блокировку, поставленную другим процессом.

**Бесконечный цикл** — блокировка будет пытаться захватить ресурс снова и снова, пока это не удастся. Это реализует «busy wait» (ожидание с повторными попытками).

In [ ]:
            # Атомарная операция: установить если нет (NX) с временем жизни (EX)
            if self.redis.set(self.lock_name, self.token, nx=True, ex=self.expire):

Это команда REDIS SET, опции:
- nx (только если Not eXist) - - установит значение, **только если ключ отсутствует**. Если ключ уже существует (блокировка занята), ничего не сделает и вернёт `None` (или `False`).
- ex (EXpiration) - время жизни ключа в секундах. Даже если процесс упадёт, блокировка сама освободится через `self.expire` секунд (избегаем «вечной» блокировки).
Если операция успешна (ключ не существовал и Redis его создал), метод `set` возвращает `True`

In [ ]:
return self

Блокировка захвачена успешно. Выходим из метода enter и возвращает self.

In [ ]:
            # Если не удалось захватить — ждем и пробуем снова
            time.sleep(self.retry_delay)

**Блокировка занята другим процессом/потоком — ждём и повторяем.**
- `self.retry_delay` — интервал ожидания в секундах
- Без этого `sleep` цикл потреблял бы 100% CPU, постоянно опрашивая Redis. `sleep` даёт процессору отдохнуть и снижает нагрузку на Redis.
- После паузы цикл повторяется — снова пытается выполнить атомарный `set nx`

In [ ]:
def __exit__(self, exc_type, exc_val, exc_tb):
        # Создаем пайплайн (транзакцию)
        pipe = self.redis.pipeline()
        try:
            # Включаем режим наблюдения за ключом.
            # Если другой клиент изменит этот ключ до вызова execute(), 
            # транзакция будет отменена.
            pipe.watch(self.lock_name)
            
            # Получаем текущее значение ключа
            current_token = pipe.get(self.lock_name)
            
            # Проверяем, что замок всё еще принадлежит нам
            if current_token == self.token:
                # Начинаем запись команд для атомарного выполнения
                pipe.multi()
                pipe.delete(self.lock_name)
                # Выполняем накопленные команды
                pipe.execute()
            else:
                # Если замок уже не наш, просто перестаем следить за ним
                pipe.unwatch()
        except Exception:
            # Если во время выполнения транзакции что-то пошло не так
            # (например, WatchError, если ключ изменился), просто игнорируем.
            # Это значит, что лок уже либо удален, либо перехвачен.
            pass

вызовется автоматически при выходе из блока with. Он должен безопасно освободить распределенную блокировку, только если она все еще принадлежит текущему владельцу.

exc_type - тип исключения, или none, если исключения не было
exc_val - объект исключения
exc_tb - traceback (стек вызовов)

В такой реализации exc_ не используются, метод просто игнорит любые исключения внутри блока

In [ ]:
pipe = self.redis.pipeline()

создает конвейер Redis. Он позволяет группировать несколько команд в 1 запрос (уменьшает задержки) разом, не дожидаясь ответа на каждую по отдельности. Также используется для создания транзакции благодаря watch, multi, execute. Транзакция нужна для того, чтобы выполнить группу команд как единое логическое действие, через multi.

In [ ]:
pipe.watch(self.lock_name)

Устанавливает наблюдателя за ключом блокировки
- Команда Redis WATCH отслеживает изменения ключа
- Если **после** вызова `WATCH` и **до** вызова `EXECUTE` какой-то другой клиент изменит (`SET`, `DEL`, `EXPIRE` и т.д.) ключ `self.lock_name`, то транзакция (`MULTI`/`EXEC`) автоматически отменится.
- Это защищает от ситуации, когда блокировка успела «протухнуть» по `expire` и её захватил другой процесс, а мы по ошибке удаляем чужую блокировку.

In [ ]:
current_token = pipe.get(self.lock_name)

**Читает текущее значение ключа блокировки.**
- Важно: команда `GET` выполняется **до** вызова `pipe.multi()`, поэтому она не входит в транзакцию, но работает под защитой `WATCH`.
- Если ключ уже удалён или истёк, `current_token` будет `None`.
- `current_token` — это строка UUID, которую записал владелец блокировки при захвате.

In [ ]:
if current_token == self.token:

**Сравнивает токен в Redis с нашим токеном.**
- Если значения совпадают — блокировка всё ещё наша, и мы можем её безопасно удалить.
- Если не совпадают или `current_token` равен `None` — блокировка либо уже истекла, либо была удалена, либо перехвачена другим процессом. В этом случае удалять её нельзя.

In [ ]:
                # Начинаем запись команд для атомарного выполнения
                pipe.multi()
                pipe.delete(self.lock_name)
                # Выполняем накопленные команды
                pipe.execute()

**Блок безопасного удаления блокировки (только если она наша):**
1. `pipe.multi()` — начинает транзакцию Redis. Все последующие команды (`delete`) будут выполняться атомарно, но только если ключ не изменился после `WATCH`.
2. `pipe.delete(self.lock_name)` — добавляет команду `DEL` в транзакцию.
3. `pipe.execute()` — отправляет транзакцию на выполнение.
    - Если ни один другой клиент не менял `lock_name` между `WATCH` и `EXECUTE`, транзакция успешно выполнится и ключ будет удалён.
    - Если кто-то изменил ключ (например, блокировка истекла и Redis сам её удалил, или другой процесс её перехватил), Redis вызовет исключение `WatchError`.

In [ ]:
            else:
                # Если замок уже не наш, просто перестаем следить за ним
                pipe.unwatch()

**Блок, если блокировка больше не принадлежит нам:**
- Вызываем `UNWATCH`, чтобы снять наблюдение за ключом (иначе соединение останется в состоянии «наблюдения», что может помешать будущим операциям)
- Никакого `DELETE` не происходит — чужие блокировки мы не трогаем.

In [ ]:
        except Exception:
            # Если во время выполнения транзакции что-то пошло не так
            # (например, WatchError, если ключ изменился), просто игнорируем.
            # Это значит, что лок уже либо удален, либо перехвачен.
            pass

**Ловим и игнорируем любые исключения.**
- Основное ожидаемое исключение — `WatchError` (ключ изменился после `WATCH`). Это означает, что блокировка уже была удалена или перехвачена другим процессом — мы просто ничего не делаем.
- Также могут быть сетевые ошибки, таймауты и т.д. В любом случае, безопаснее **не пытаться повторно удалять** блокировку, чтобы случайно не удалить чужую.
- `pass` означает «ничего не делаем» — для метода `__exit__` это нормально, он не обязан возвращать какое-то значение.


In [ ]:
r_client = redis.Redis(host='localhost', port=6379, decode_responses=True)

**Создаёт экземпляр клиента Redis для подключения к серверу Redis.**
 `host='localhost'`
**Адрес сервера Redis.**
- `'localhost'` (или `'127.0.0.1'`) означает, что Redis запущен на той же машине, где выполняется Python-код.

`port=6379`
**Порт сервера Redis.**
- `6379` — это **стандартный порт Redis** по умолчанию.

 `decode_responses=True`
**Автоматически декодировать ответы Redis из байтов в строки.**
- Redis возвращает данные в виде байтов (`b'value'`).
- При `decode_responses=True` клиент автоматически преобразует байты в строки Python (`'value'`).
**Почему это важно для вашего кода с блокировками?**
- В методах `__enter__` и `__exit__` вы работаете с токенами как со строками (`self.token = str(uuid.uuid4())`).
- Без `decode_responses=True` сравнение `current_token == self.token` могло бы не работать, потому что `current_token` был бы байтовой строкой (`b'...'`), а `self.token` — обычной строкой (`'...'`).

uuid - универсально уникальный идентификатор, состоит из 128 бит.


In [ ]:
mu = RedisLock(r_client, "my_counter_lock")
result = 0

**Создаёт экземпляр распределённой блокировки Redis.**
- r_client. Redis клиент, который создали ранее
- my_counter_lock. Строка. Уникальное имя блокировки в Redis.
**Инициализирует переменную `result` нулевым значением.**


In [ ]:
def function():
    with mu:  # Точка синхронизации
        # Читаем -> Ждем -> Пишем
        global result
        r = result
        time.sleep(0.1)  # Имитация долгой работы
        result = r + 1

 `global result`
**Объявляет, что `result` — глобальная переменная.**
- Без этого ключевого слова Python создал бы локальную переменную внутри функции
- Здесь мы собираемся **изменять** глобальный `result`, поэтому `global` обязателен

 `with mu: # Точка синхронизации`
**Вход в критическую секцию с распределённой блокировкой Redis.**
- `mu` — это объект `RedisLock`, созданный ранее
- Только один процесс/поток может находиться внутри этого блока одновременно
- Остальные процессы будут ждать в цикле `while True` внутри `__enter__`
- with автоматически выполняет код при входе в блок и при выходе из него, он сам считывает, какой сейчас этап и выполняет нужный метод.

 `r = result`
**Читаем текущее значение глобальной переменной в локальную переменную.**

`time.sleep(0.1) # Имитация долгой работы`
**Имитация длительной обработки (0.1 секунды).**
- **Ключевая проблема:** блокировка Redis удерживается во время этого сна
- За это время другие процессы не могут войти в `with` блок (ждут)
- Но сам Redis не блокируется — он просто хранит ключ `"my_counter_lock"` с TTL

`result = r + 1`
**Увеличиваем значение на 1 и сохраняем обратно в глобальную переменную.**


In [ ]:
def main():
    global result
    result = 0
    print("Запуск потоков...")
    with ThreadPoolExecutor(max_workers=5) as executor:
        futures = [executor.submit(function) for _ in range(10)]
        for f in futures:
            f.result() # Ожидаем завершения всех потоков

`result = 0`
**Инициализирует глобальную переменную `result` нулём.**
- Теперь `result` доступна всем потокам для чтения и записи

``with ThreadPoolExecutor(max_workers=5) as executor:
**Создаёт пул потоков с помощью контекстного менеджера.**
Разбор:
- `ThreadPoolExecutor` — класс из `concurrent.futures`, управляющий пулом потоков
- `max_workers=5` — максимальное количество потоков, которые могут выполняться **одновременно**
- `with ... as executor:` — при входе создаётся пул, при выходе (после блока) автоматически вызывается `executor.shutdown(wait=True)`, который дожидается завершения всех задач и освобождает ресурсы
- `executor` — объект, через который мы отправляем задачи в пул
**Почему 5 потоков, а задач 10?** Пул будет выполнять по 5 задач параллельно, остальные 5 будут ждать своей очереди.

`futures = [executor.submit(function) for _ in range(10)]`
**Создаёт 10 задач и отправляет их в пул потоков.**
Разбор:
- `executor.submit(function)` — отправляет функцию `function` на выполнение в пуле. Возвращает объект **`Future`** (обещание получить результат в будущем)
- `for _ in range(10)` — повторяет 10 раз (переменная `_` означает "мне не нужно значение счётчика")
- `[...]` — список comprehension, создаёт список из 10 объектов `Future`
**Что происходит:** 5 задач сразу начинают выполняться в 5 потоках, остальные 5 ждут в очереди. Каждая задача вызывает `function()`, которая увеличивает `result` на 1.

`for f in futures: f.result()`
**Ожидает завершения всех задач.**
Разбор:
- `for f in futures` — итерируем по всем объектам `Future` (в порядке создания)
- `f.result()` — блокирует текущий поток (главный поток `main`) до тех пор, пока задача `f` не завершится
- Если задача завершилась с исключением, `f.result()` пробросит его в главный поток
**Зачем это нужно?** Без этого главный поток мог бы завершиться раньше, чем закончат работу все задачи, и программа закрылась бы (или `with` вызвал бы `shutdown`, но с этим вызовом мы контролируем порядок).


In [ ]:
if __name__ == "__main__":
    try:
        main()
    except redis.exceptions.ConnectionError:
        print("Ошибка: Не удалось подключиться к Redis. Убедитесь, что сервер запущен.")

`except redis.exceptions.ConnectionError:`
**Перехватывает специфическое исключение — ошибку подключения к Redis.**
 Что такое `redis.exceptions.ConnectionError`:
- Это исключение из библиотеки `redis`
- Возникает, когда клиент не может подключиться к серверу Redis
- Примеры ситуаций:
    - Redis сервер не запущен
    - Неправильный хост/порт
    - Сетевые проблемы
    - Брандмауэр блокирует соединение